# Differentiable segregation dynamics — the PyTorch twin

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JoelCC64/segregation-dynamics/blob/main/notebooks/differentiable_segregation.ipynb)

Phase 2 of the [segregation-dynamics](https://github.com/JoelCC64/segregation-dynamics) project.
The [Fortran reference implementation](https://github.com/JoelCC64/segregation-dynamics/tree/main/fortran) answers *"what does this rule set produce?"* — fast.
This notebook builds its **differentiable twin** and demonstrates the two capabilities that
differentiability unlocks and that the forward paradigm cannot easily provide:

1. **Sensitivity analysis** — exact $\partial(\text{segregation})/\partial(\text{parameters})$ *through the dynamics*, by autodiff;
2. **Gradient-based calibration** — recovering an unknown tolerance parameter from an observed trajectory (an inverse problem).

Runs on the free Colab GPU runtime (`Runtime → Change runtime type → T4 GPU`) **or on CPU** —
every cell here completes in a few minutes either way. Model code lives in
[`pytorch/segregation_torch.py`](https://github.com/JoelCC64/segregation-dynamics/blob/main/pytorch/segregation_torch.py) (single source of truth; this notebook only drives it).

## 1 · The core challenge: gradients through discrete decisions

Agent-based models are notoriously hard to calibrate because they are built from
non-differentiable pieces. This model has three:

| Where | Discrete object | Why gradients die |
|---|---|---|
| neighbourhood | $\mathbb{1}[d_{ij} \le R]$ | piecewise constant in positions and in $R$ |
| happiness | $\mathbb{1}[n_i \ge K_{\min}]$ | Heaviside step in count and threshold |
| kinetics | unhappy agents teleport at random until happy | discrete accept/reject of random proposals |

**The relaxation used here.** Soft disk membership, a soft happiness gate, and — replacing
the random teleports — **unhappiness-gated gradient ascent** of each agent's own soft
same-type count:

$$n_i=\sum_{j\,:\,t_j=t_i}\sigma\!\left(\frac{R-d_{ij}}{\tau_d}\right),\qquad
h_i=\sigma\!\left(\frac{n_i-K_{\min}}{\tau_h}\right),\qquad
x_i \leftarrow \operatorname{clip}\!\Big(x_i+\eta\,(1-h_i)\,\nabla_{x_i} n_i\Big).$$

The ascent direction has a closed form — attraction toward same-type agents, dominated by the
*marginal neighbours* near the disk edge where $\sigma'$ peaks. Every arrow is smooth, so any
observable at any time is differentiable w.r.t. $K_{\min}$, $R$, $\tau$, $\eta$ by backprop
through the unrolled dynamics. As $\tau_d,\tau_h\to 0$ the *rules* converge to the discrete
ones (verified in the smoke test below).

**Alternatives considered, and why not.**
*Gumbel-softmax / straight-through* over candidate relocation sites would preserve the teleport
kinetics, but a softmax-weighted mixture of positions is not a valid configuration of this model,
and hard sampling with straight-through gives biased gradients — a shaky foundation for
calibration. A *density-field (PDE) formulation* is naturally differentiable but abandons the
agent picture altogether. The gated gradient flow keeps agents and utilities and yields
**exact gradients of an approximate model**, rather than approximate gradients of the exact model.

**What is honestly different:** the kinetics. Relaxed agents drift toward the nearest same-type
crowd; the Fortran teleporters can jump anywhere. Both reach full happiness at the reference
density, but final segregation differs quantitatively (≈ 0.59 vs 0.629 — measured in §2). The
right mental model is a *differentiable twin validated against the reference*, not a bit-exact
replacement. Full design discussion: [`pytorch/README.md`](https://github.com/JoelCC64/segregation-dynamics/blob/main/pytorch/README.md).

In [ ]:
# Setup: device detection + robust import (Colab clones the repo on first run).
import os, subprocess, sys, time
import torch

try:
    import segregation_torch
except ImportError:
    for cand in ("../pytorch", "segregation-dynamics/pytorch"):
        if os.path.exists(os.path.join(cand, "segregation_torch.py")):
            sys.path.insert(0, cand)
            break
    else:
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/JoelCC64/segregation-dynamics.git"],
                       check=True)
        sys.path.insert(0, "segregation-dynamics/pytorch")
    import segregation_torch

from segregation_torch import (Config, init_state, soft_counts, soft_happiness,
                               hard_metrics, simulate, default_device)

device = default_device()

def sync():
    """Wait for pending GPU kernels so wall-clock timings are honest."""
    if device.type == "cuda":
        torch.cuda.synchronize()

print(f"torch {torch.__version__} | device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# ---------------------------------------------------------------- SMOKE TEST
# Small system (N = 100), runs in well under a minute on CPU. Verifies the
# machinery end to end BEFORE you spend GPU time: shapes, the discrete limit,
# gradient checkpointing, and — crucially — that autodiff gradients through
# the unrolled dynamics match finite differences to near machine precision.
t0 = time.time()
cfg_s = Config.scaled(50, n_steps=15, seed=7)      # N = 100, box ≈ 22.4

X, types = init_state(cfg_s, device=device)        # X: (100, 2), types: (100,)
assert X.shape == (100, 2) and types.shape == (100,)
assert 0.0 <= X.min().item() and X.max().item() <= cfg_s.box
n_same, n_total = soft_counts(X, types, cfg_s.radius, cfg_s.tau_d)   # (100,), (100,)
assert n_same.shape == (100,) and (n_same <= n_total + 1e-6).all()
h = soft_happiness(n_same, cfg_s.k_min, cfg_s.tau_h)
assert ((h > 0) & (h < 1)).all()
print("shapes and ranges                      OK")

# tau_d -> 0 recovers the integer neighbour counts of the Fortran model
ns_sharp, _ = soft_counts(X, types, cfg_s.radius, 1e-4)
d2 = ((X[:, None, :] - X[None, :, :]) ** 2).sum(-1)
inside = d2 <= cfg_s.radius ** 2
inside.fill_diagonal_(False)
ns_hard = (inside & (types[:, None] == types[None, :])).sum(1)
assert (ns_sharp - ns_hard.to(ns_sharp.dtype)).abs().max() < 1e-3
print("tau_d -> 0 recovers discrete counts    OK")

# autodiff through the dynamics == finite differences (float64)
X64, ty64 = init_state(cfg_s, device=device, dtype=torch.float64)
k = torch.tensor(cfg_s.k_min, dtype=torch.float64, device=device, requires_grad=True)
simulate(X64, ty64, cfg_s, k_min=k)["seg"][-1].backward()
g_auto = k.grad.item()
eps = 1e-4
with torch.no_grad():
    Sp = simulate(X64, ty64, cfg_s, k_min=torch.tensor(cfg_s.k_min + eps, dtype=torch.float64, device=device))["seg"][-1].item()
    Sm = simulate(X64, ty64, cfg_s, k_min=torch.tensor(cfg_s.k_min - eps, dtype=torch.float64, device=device))["seg"][-1].item()
g_fd = (Sp - Sm) / (2 * eps)
rel = abs(g_auto - g_fd) / max(abs(g_fd), 1e-12)
assert rel < 1e-4, f"autodiff vs FD mismatch: {rel:.1e}"
print(f"autodiff vs finite differences         OK (rel err {rel:.1e})")

# checkpointed gradients must equal plain gradients (same graph, recomputed)
g = {}
for ck in (False, True):
    k = torch.tensor(cfg_s.k_min, device=device, requires_grad=True)
    simulate(X, types, cfg_s, k_min=k, use_checkpoint=ck)["seg"][-1].backward()
    g[ck] = k.grad.item()
assert abs(g[True] - g[False]) <= 1e-4 * max(abs(g[False]), 1e-12)
print(f"checkpointed gradients exact           OK (dS/dK_MIN = {g[False]:+.3e})")

sync()
print(f"\nSMOKE TEST PASSED in {time.time() - t0:.1f} s — safe to scale up.")

## 2 · Forward dynamics at reference scale

Same scale as the Fortran run: $N = 2000$ agents, box $L = 100$, radius $R = 10$, threshold
$K_{\min} = 30$. Forward-only inference keeps no autodiff graph, so memory is one $O(N^2)$
kernel per step. The **hard** metrics below evaluate the *exact discrete definitions* (the
Fortran ones) on the relaxed trajectory — an apples-to-apples comparison with the reference:
the Fortran teleport run goes from segregation 0.498 (statistically perfect mixing) to 0.629
at convergence, with the happy fraction reaching 1.0.

In [ ]:
# Forward run at the Fortran reference scale (defaults of Config).
cfg = Config(n_steps=200)                     # N = 2000, box = 100, seed 20260707
X0, types = init_state(cfg, device=device)

sync(); t0 = time.time()
with torch.no_grad():
    run = simulate(X0, types, cfg, record_every=5)
sync()
print(f"N = {cfg.n_a + cfg.n_b}, T = {cfg.n_steps} steps: {time.time() - t0:.1f} s on {device}")

fh0, s0 = hard_metrics(X0, types, cfg.radius, cfg.k_min)
fhT, sT = hard_metrics(run["X"], types, cfg.radius, cfg.k_min)
print(f"hard fraction happy : {fh0:.3f} -> {fhT:.3f}    (Fortran teleport: 0.483 -> 1.000)")
print(f"hard segregation    : {s0:.3f} -> {sT:.3f}    (Fortran teleport: 0.498 -> 0.629)")
print("same happiness convergence; milder segregation — local drift joins the")
print("nearest crowd, teleports nucleate denser clusters (see §5).")

In [ ]:
import matplotlib.pyplot as plt

Xf, X0c, tyc = run["X"].cpu(), X0.cpu(), types.cpu()
fig, axes = plt.subplots(1, 2, figsize=(11, 5.4), sharex=True, sharey=True)
for ax, P, title in ((axes[0], X0c, "initial — statistically mixed"),
                     (axes[1], Xf, f"after {cfg.n_steps} relaxation steps")):
    for t, c in ((1, "tab:blue"), (0, "tab:red")):
        m = tyc == t
        ax.scatter(P[m, 0], P[m, 1], s=4, c=c, alpha=0.6, linewidths=0)
    ax.set_title(title)
    ax.set_aspect("equal")
    ax.set_xlim(0, cfg.box); ax.set_ylim(0, cfg.box)
plt.tight_layout(); plt.show()

fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
ax[0].plot(run["steps"], run["seg"].cpu(), label="soft (differentiable)")
ax[0].axhline(0.629, ls="--", c="gray", label="Fortran final (teleport)")
ax[0].set_xlabel("step"); ax[0].set_ylabel("segregation index"); ax[0].legend()
ax[1].plot(run["steps"], run["frac_happy"].cpu(), c="tab:green")
ax[1].set_xlabel("step"); ax[1].set_ylabel("soft fraction happy")
plt.tight_layout(); plt.show()

## 3 · Demo 1 — sensitivity analysis *through* the dynamics

What is $\partial S_T/\partial K_{\min}$: how much does the segregation reached after $T$
steps change per unit of tolerance threshold? With the Fortran model this needs noisy finite
differences over ensembles — one full simulation per probe point per seed. Here it is **one
backward pass**. Two things to notice:

- **the sign:** $\partial S/\partial K_{\min} > 0$ — more demanding agents produce *more*
  segregation. Schelling's core insight, delivered as a computed number with error bars from
  autodiff exactness rather than sampling.
- **the validation:** central finite differences converge to the autodiff value as
  $\delta \to 0$ (float64 table below). At large $\delta$ they *disagree* — not an autodiff
  bug but a real property of gradients through long unrolled dynamics: the map
  $K_{\min}\mapsto S_T$ develops fine-scale structure as $T$ grows (clipping kinks,
  trajectory divergence). We verified the same check at $T{=}50$ agrees at all $\delta$.
  Stating this caveat is standard good practice in differentiable simulation.

In [ ]:
# Exact parameter sensitivities by backprop through 100 steps of dynamics.
# float64 so the delta-study below is meaningful; the workload is small enough
# that the fp64 penalty on consumer GPUs is irrelevant here.
X64, ty64 = init_state(cfg, device=device, dtype=torch.float64)
k_min = torch.tensor(cfg.k_min, dtype=torch.float64, device=device, requires_grad=True)
radius = torch.tensor(cfg.radius, dtype=torch.float64, device=device, requires_grad=True)

sync(); t0 = time.time()
out = simulate(X64, ty64, cfg, k_min=k_min, radius=radius,
               n_steps=100, record_every=100, use_checkpoint=True)
S = out["seg"][-1]
S.backward()                       # <- gradients through the whole trajectory
sync()
gk, gr = k_min.grad.item(), radius.grad.item()
print(f"S(T=100)   = {S.item():.4f}          [{time.time() - t0:.0f} s incl. backward]")
print(f"dS/dK_MIN  = {gk:+.4e}    more demanding agents -> MORE segregation")
print(f"dS/dR      = {gr:+.4e}    larger neighbourhoods dilute the same-type share")

# Validation: central differences converge to the autodiff value as delta -> 0.
def S_at(kval):
    with torch.no_grad():
        kk = torch.tensor(kval, dtype=torch.float64, device=device)
        return simulate(X64, ty64, cfg, k_min=kk, n_steps=100,
                        record_every=100)["seg"][-1].item()

print("\n   delta       FD dS/dK_MIN     |FD - autodiff| / |autodiff|")
for delta in (1e-3, 1e-5):
    g_fd = (S_at(cfg.k_min + delta) - S_at(cfg.k_min - delta)) / (2 * delta)
    print(f"   {delta:.0e}    {g_fd:+.6e}        {abs(g_fd - gk) / abs(gk):.1e}")

## 4 · Demo 2 — gradient-based calibration (the inverse problem)

Scenario: we observe a segregation *trajectory* produced by some unknown tolerance
$K^{\ast}$ and must find the parameter that reproduces it. Classically this means a parameter
sweep — dozens of forward ensembles. With the differentiable twin it is gradient descent on

$$\mathcal{L}(K)=\big\lVert S_{0:T}(K)-S^{\mathrm{obs}}_{0:T}\big\rVert^2 ,$$

backpropagating **through the entire unrolled simulation** (120 steps here). Gradient
checkpointing keeps memory at $O(N^2)$ instead of $O(T\,N^2)$ — activations are recomputed
during the backward pass (~2× forward compute, verified bit-exact in the smoke test).

The target is generated with $K^{\ast} = 35$ (hidden from the optimiser); the search starts
deliberately far away at $K_0 = 24$. Runtime: ~1 min on CPU, seconds on a T4.

In [ ]:
# Gradient-based recovery of a hidden K_MIN from an observed trajectory.
cfg_c = Config.scaled(300, n_steps=120, seed=42)     # N = 600 at reference density
X0c, types_c = init_state(cfg_c, device=device)

K_TRUE = 35.0                                        # hidden "ground truth"
with torch.no_grad():
    target = simulate(X0c, types_c, cfg_c, record_every=5,
                      k_min=torch.tensor(K_TRUE, device=device))["seg"]

k_hat = torch.tensor(24.0, device=device, requires_grad=True)   # wrong on purpose
opt = torch.optim.Adam([k_hat], lr=0.5)
sched = torch.optim.lr_scheduler.StepLR(opt, step_size=20, gamma=0.4)

history = []
sync(); t0 = time.time()
for it in range(60):
    opt.zero_grad()
    sim = simulate(X0c, types_c, cfg_c, k_min=k_hat, record_every=5,
                   use_checkpoint=True)["seg"]
    loss = ((sim - target) ** 2).mean()
    loss.backward()                    # <- through all 120 steps
    opt.step()
    sched.step()
    with torch.no_grad():
        k_hat.clamp_(5.0, 55.0)        # physically plausible range
    history.append((loss.item(), k_hat.item()))
    if it % 10 == 0 or it == 59:
        print(f"iter {it:2d}   loss {loss.item():.3e}   K_MIN {k_hat.item():6.2f}")
sync()
print(f"\nrecovered K_MIN = {k_hat.item():.2f}   (true {K_TRUE}, started at 24.0)"
      f"   [{time.time() - t0:.0f} s]")

In [ ]:
loss_hist = [l for l, _ in history]
k_hist = [kk for _, kk in history]

fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
ax[0].plot(k_hist, marker=".", lw=1)
ax[0].axhline(K_TRUE, ls="--", c="gray", label=f"true $K^*$ = {K_TRUE}")
ax[0].set_xlabel("Adam iteration"); ax[0].set_ylabel(r"$\hat{K}_{\min}$"); ax[0].legend()
ax[1].semilogy(loss_hist)
ax[1].set_xlabel("Adam iteration"); ax[1].set_ylabel("trajectory MSE")
plt.tight_layout(); plt.show()

## 5 · Two paradigms, one model

| | **Fortran 95** (`/fortran`) | **PyTorch** (this notebook) |
|---|---|---|
| Question answered | *what does this rule set produce?* | *which parameters produce what we observe, and how sensitive is the outcome?* |
| Discreteness | exact — hard disks, hard thresholds, teleports | relaxed — soft kernels, soft gates, gradient flow |
| Gradients | none (finite differences over ensembles) | exact w.r.t. any parameter, one backward pass |
| Forward cost | ~0.4 s for the full converged run, single core | seconds on a GPU; $O(N^2)$ memory |
| Strengths | ensembles, parameter sweeps, long runs, HPC pipelines | calibration, sensitivity, integration with ML |
| Weaknesses | inverse problems only by brute force | kinetics approximated; $O(N^2)$ pairwise tensors |

Neither replaces the other. The Fortran code **is** the model; the PyTorch twin is the
*instrument that lets gradients interrogate it*. The cross-checks — hard metrics evaluated on
relaxed trajectories, the $\tau \to 0$ limit, the finite-difference tables — are what make
either of them trustworthy.

**Known limitations, stated plainly**

- Relaxed kinetics ≠ teleport kinetics: quantitative endpoints differ (hard segregation
  ≈ 0.59 here vs 0.629 in Fortran). Only the *rules* converge as $\tau \to 0$, not the transport.
- `clamp` at the walls and the per-step cap create gradient dead zones / kinks.
- $O(N^2)$ pairwise tensors cap $N$ around $10^4$ on a T4; cell lists (as the Fortran code
  uses) or KeOps-style lazy tensors would lift this.
- Gradients through long unrolls probe increasingly fine structure of the loss landscape
  (§3's $\delta$-study) — trajectory-matching losses over moderate horizons are the robust choice.

**Phase 3 candidates:** joint calibration of $(K_{\min}, R)$ from noisy or partial
observations; sensitivity maps along trajectories; $\tau$-annealing within a run; comparing
against ensemble finite differences from the Fortran side.